In [ ]:
import pandas as pd
from rapidfuzz import fuzz
from itertools import combinations 

df = pd.read_csv("patients.csv")

def normalize_phone(phone):
    return phone.replace("-","")

def birth_date_variants(date_str):
    """월/일 뒤바뀜까지 고려한 후보 형태 반환"""
    parts = date_str.split("-")
    if len(parts) !=3:
        return {date_str}
    y, m, d = parts
    return {date_str, f"{y}-{d}-{m}"} # 원본 + 월일 뒤바뀐 버전

def match_score(row1,row2):
    score=0
    # 이름 유사도 (0~100)
    score += fuzz.ratio(row1["name"],row2["name"]) * 0.3

    # 생년월일: 원본이거나 월일 뒤바뀐 버전이 일치하면 만점
    if row2["birth_date"] in birth_date_variants(row1["birth_date"]):
        score += 100 * 0.3
    else:
        score += fuzz.ratio(row1["birth_date"],row2["birth_date"]) * 0.3

    #전화번호 : 하이픈 제거 후 비교
    if normalize_phone(row1["phone"]) == normalize_phone(row2["phone"]):
        score += 100 * 0.25
    else:
        score += fuzz.ratio(normalize_phone(row1["phone"]),normalize_phone(row2["phone"])) * 0.25

    #주소 유사도
    score += fuzz.token_sort_ratio(row1["address"], row2["address"]) * 0.15

    return score

# 같은 이름을 가진 그룹 안에서만 비교 (전체 조합은 너무 많으니까)        
candidates = []
for name,group in df.groupby("name"):
    if len(group) < 2:
        continue
    for (i1,r1), (i2,r2) in combinations(group.iterrows(),2):
        s = match_score(r1,r2)
        candidates.append((r1["patient_id"], r2["patient_id"], round(s, 1)))

result = pd.DataFrame(candidates, columns=["id1", "id2", "score"]).sort_values("score", ascending=False)
print(result.head(20))
